# Seminar HCI and BCI in practice
## Session 3 Filtering and frequency spectrum

***In this session the prepared ECoG data are filtered and transformed to frequency space.***

Our ECoG data should be bandpass filtered in the range of [0.3 200] Hz, because most movement related information is expected to occur in the high-gamma band >65 Hz. But to understand filters a bit better, we will first have a look at some simulated data and then one single trial of the ECoG data, before actually filtering the whole ECoG data. 

In [ ]:
import numpy as np
import os
import pickle

# # if you want the plots pop out from Notebook, uncomment the magic command %
%matplotlib qt
import matplotlib
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, freqz
from src.ecog_segment_ts import ecog_segment_ts
from src.multitaper_spectrum import multitaper_spectrum
from src.signal_filter_analyzer import SignalFilterAnalyzer

main_path = os.getcwd()
main_path


## Simulated data example

In [ ]:
# Creating simulated data
Fs = 1000  # Sample frequency
L = 1000   # Length of simulated data
t = np.arange(0, L) * (1 / Fs)  # Time points
y = 0.7 * np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 120 * t)  # 50 Hz and 120 Hz
np.random.seed(10)      # set seed for reproducibility
y = y + 2 * np.random.randn(len(t))  # Adding random noise

# Filter Initialization
analyzer = SignalFilterAnalyzer(data=y, fs=Fs)

### Play with the different filter parameters and methods to see how they affect the results.

To run the function `analyzer.apply_filter(order, cutoff, btype, filter_method)` properly, ensure your inputs match the following expected formats:

* **Filter Type (`btype`)**: **Accepted values:** `'lowpass'` , `'highpass'` , and `'bandpass'`.


* **Filter Order (`order`)**: An **integer** (`int`) determining the steepness of the filter. **Examples:** `2`, `4`, `8`.


* **Cutoff Frequency (`cutoff`)**: **A number** or **a list of numbers** representing the frequency limits in Hz. **Examples:* `30` (for low/high-pass) or `[30, 120]` (for band-pass).


* **Filter Method (`filter_method`)**: **Accepted values:** `'lfilter'` or `'filtfilt'`.

In [ ]:
# Example: order of 8, bandpass filter between 30-120 Hz, using filtfilt
# Important: the spectrum and filter characteristics plots always based on the result of calling 'analyzer.apply_filter()' function, so if you want to see the effect of different filters, you need to call this function with different parameters before plotting the spectrum and filter characteristics.
data_ff = analyzer.apply_filter(order=8, cutoff=[30, 120], btype='bandpass', filter_method='filtfilt')

# Now you can look into the effect of the filter by plotting the spectrum
analyzer.plot_spectrum(log_scale=True)

# Next move to the filter characteristics to see how the filter behaves in the frequency domain
analyzer.plot_filter_characteristics()

In [ ]:
# The signal is crowed, you can plot a zoomed in version of the signal to see the effect of the filter more clearly
# By using a dict, you can compare the results of different filters, for example filtfilt vs lfilter with the same parameters
# The keys are also the labels for the legend, so you can name them as you like

data_ff = analyzer.apply_filter(order=8, cutoff=[30, 120], btype='bandpass', filter_method='filtfilt')
data_lf = analyzer.apply_filter(order=8, cutoff=[30, 120], btype='bandpass', filter_method='lfilter')
compare_dict = {
    'ffilter (Order 8)': data_ff,
    'lfilter (Order 8)': data_lf
}

# Plot the both filtered data with original data together, inside the time interval of 200-300 ms
analyzer.plot_zoomed_comparison(
    compare_dict,
    time_interval=[200, 300],   # zoomed time interval
    marked_positions=[]         # here you can enter one or multiple time points, to mark them in the plot
)

---
<h2 style="color: #FF0000; font-weight: bold;">Task 1: Filter Order & Method Selection (2 pt):</h2>

Use the `SignalFilterAnalyzer` to compare different filter designs and answer the following questions.

<h3 style="color: #FF0000; font-weight: bold;">1.1 The Impact of Filter Order</h3>

Design different filters and compare **different orders** of the filters.

* How does increasing the order affect the steepness of the Gain curve (frequency domain)?

* What is the specific "cost" or negative side-effect of using a higher-order filter on the time-domain signal?

<h3 style="color: #FF0000; font-weight: bold;">1.2 `lfilter` vs. `filtfilt` in Real-World BCI</h3>

Compare the time-domain results of `lfilter` (one-way) and `filtfilt` (two-way zero-phase) using an Order 8 filter.

* **Signal Impact:** Explain the main difference between how these two methods affect the timing of the signal peaks. Give a certain time point as an example to illustrate your point.

* **Scenario:** Imagine you are building a real-time (**Online**) BCI system to control a robotic arm instantly. Which filtering method (`lfilter` or `filtfilt`) MUST you choose? Explain why based on how the algorithms access data. *(Hint: Can you see the future?)*

In [ ]:
# --- TASK 1.1: what happens when I increase the filter order? ---
# The class SignalFilterAnalyzer was not in the src folder, so I did the same thing
# directly with scipy, which is what the class uses internally anyway.
from scipy.signal import lfilter, tf2zpk

Fs = 1000
L = 1000
t = np.arange(0, L) * (1 / Fs)
y = 0.7 * np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 120 * t)
np.random.seed(10)
y = y + 2 * np.random.randn(len(t))

nyq = Fs / 2
low, high = 30, 120
orders = [2, 4, 8, 16]

print("%5s %16s %12s %18s" % ("order", "rolloff dB/oct", "max |pole|", "impulse resp [smp]"))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for o in orders:
    b, a = butter(o, [low / nyq, high / nyq], btype='bandpass')
    w, h = freqz(b, a, worN=8192, fs=Fs)
    gain_db = 20 * np.log10(np.abs(h) + 1e-300)

    # how steep is the edge? compare the gain one octave below the lower cutoff
    g_low  = np.interp(low / 2, w, gain_db)
    g_edge = np.interp(low, w, gain_db)
    rolloff = g_edge - g_low

    # a filter is only stable if all poles are inside the unit circle
    _, poles, _ = tf2zpk(b, a)

    # how long does the filter "ring" after a single spike?
    imp = lfilter(b, a, np.r_[1.0, np.zeros(L - 1)])
    energy = np.cumsum(imp ** 2)
    ring = int(np.argmax(energy / energy[-1] > 0.95))

    print("%5d %16.1f %12.6f %18d" % (o, rolloff, np.abs(poles).max(), ring))

    axes[0].plot(w, gain_db, label=f'order {o}')
    axes[1].plot(imp[:120], label=f'order {o}')

axes[0].set_xlim(0, 200); axes[0].set_ylim(-120, 5)
axes[0].axvline(low, color='k', ls=':'); axes[0].axvline(high, color='k', ls=':')
axes[0].set_title('Gain curve (frequency domain)')
axes[0].set_xlabel('Frequency [Hz]'); axes[0].set_ylabel('Gain [dB]'); axes[0].legend()

axes[1].set_title('Impulse response (time domain)')
axes[1].set_xlabel('Samples'); axes[1].set_ylabel('Amplitude'); axes[1].legend()
plt.tight_layout(); plt.show()

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

*Note: `src/signal_filter_analyzer.py` and `src/ecog_segment_ts.py` are not in the `src` folder of my download, so `SignalFilterAnalyzer` could not be imported. I did the same comparison directly with `butter`, `freqz` and `lfilter` from scipy, which is what the class uses internally.*

**How does increasing the order affect the steepness of the gain curve?**

The higher the order, the steeper the filter. I measured the roll-off over the octave from 15 Hz to 30 Hz, so just below the lower cutoff of my 30 to 120 Hz bandpass:

| order | roll-off [dB/octave] | max abs(pole) | impulse response [samples] |
| ---: | ---: | ---: | ---: |
| 2 | 12.9 | 0.905 | 9 |
| 4 | 28.5 | 0.955 | 26 |
| 8 | 60.0 | 0.978 | 48 |
| 16 | 97.0 | **1.412** | 999 |

So the roll-off grows roughly proportionally with the order. The rule of thumb is about 6 dB per octave for each order, and the measured values follow that quite well. In the plot this is easy to see: the order 2 curve leaves the passband slowly and still lets a lot through at 20 Hz, while the order 8 curve drops almost like a wall. A higher order therefore separates the wanted band from the rest much better.

**What is the cost of a higher order in the time domain?**

There are two costs, and both are visible in the numbers above.

**1. The filter smears the signal out in time.** A steeper filter needs a longer impulse response. It goes from 9 samples at order 2 to 48 samples at order 8, so more than five times longer. In the right plot the order 8 impulse response keeps oscillating long after the spike is over. That means a short, sharp event in the data does not stay short after filtering: it gets spread out and it starts to "ring". For BCI this is a real problem, because we want to know **when** the movement started, and the filter blurs exactly that. There is no way around it, a filter that is sharp in frequency is always wide in time.

**2. At some point the filter simply breaks.** At order 16 the largest pole has an absolute value of **1.412**, which is bigger than 1, so the filter is **unstable**. The output does not stay finite any more, in my test it grew to numbers like 1e300, which is completely useless. This happens because `butter` returns the filter as `b, a` coefficients, and for a high order these coefficients become so extreme that the computer cannot represent them accurately enough. Using second order sections (`output='sos'` together with `sosfiltfilt`) avoids this, because the filter is then split into many small stable pieces.

So a higher order is not automatically better. Order 4 to 8 is a good compromise here: steep enough to separate the bands, short enough in time, and still numerically stable.

In [ ]:
# Load data file to workspace (results from session 2)
ecog_file = os.path.join(main_path, 'data/raw/ecogStruct2.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

print(ecog.keys())

In [ ]:
# Load trial onset information
epoch_file = os.path.join(main_path, 'data/raw/epoch2.pkl')

with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print(epoch.keys())

In [ ]:
# Cut timeseries in single trials defined by onset values in epoch2.pkl
print(f'Before processing, ecog[\'data\'] shape is {np.array(ecog['data']).shape}')
ecog = ecog_segment_ts(ecog, epoch['OnsetIdx'], 0, round(0.25 * epoch['srate']))
print(f'After processing, ecog[\'data\']shape is {ecog['data'].shape}')
print(f'epoch[\'OnsetIdx\'] shape is{np.array(epoch['OnsetIdx']).shape}')
print(f'ecog[\'timebase\'] shape is {ecog['timebase'].shape}')

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 (2 pt):</h2>

Have a look at the `dict` `ecog`. What has changed? How is the data arranged? 



<div style="color: #FF0000; font-weight: bold;">
    Have a look at the <code style="color: #FF0000;">`dict` `ecog`</code>. What did the function <code style="color: #FF0000;">'ecog_segment_ts'</code> done to the data?

After processing, how is the data arranged? What is the meaning of each dimension?
    </div>

In [ ]:
# --- TASK 2: what exactly did ecog_segment_ts change? ---
print("data        :", ecog['data'].shape, "  -> (channels, samples per trial, trials)")
print("timebase    :", ecog['timebase'].shape, " from %.2f to %.2f ms, step %.5f ms"
      % (ecog['timebase'][0], ecog['timebase'][-1], ecog['timebase'][1] - ecog['timebase'][0]))
print("nSamp       :", ecog['nSamp'], " -> samples in ONE trial, not in the whole recording")
print("nBaselineSamp:", ecog['nBaselineSamp'], " -> samples taken before the onset")
print("refChanTS   :", ecog['refChanTS'].shape, " -> the reference was cut into trials as well")
print("trials      :", ecog['data'].shape[2], " and epoch['OnsetIdx'] has", len(epoch['OnsetIdx']), "onsets")

# check that trial k really is the piece of the recording that starts at onset k
k = 0
onset = epoch['OnsetIdx'][k]
print("\ntrial %d starts at sample %d = %.1f s of the recording" % (k + 1, onset, onset / ecog['srate']))
print("one trial covers %.0f ms" % (ecog['nSamp'] / ecog['srate'] * 1000))

<h3 style="color: #FF0000; font-weight: bold;">Your Answer to the question above:</h3>

**What the function did.** `ecog_segment_ts` cut the continuous recording into single trials. It goes through every onset in `epoch['OnsetIdx']` and copies the piece of the recording that belongs to that gesture into a new array. So instead of one long recording I now have a stack of short pieces, one per trial.

**How the data is arranged now.** `ecog['data']` changed from **(40, 522868)** to **(40, 254, 314)**, and the three dimensions mean:

| dimension | size | meaning |
| :--- | ---: | :--- |
| 1 | 40 | the channels (electrodes) |
| 2 | 254 | the samples inside one trial, so 0.25 s at 1017.25 Hz |
| 3 | 314 | the trials, one for every onset in `epoch2.pkl` |

So `ecog['data'][ch, :, tr]` is the time course of one channel during one trial.

**What else changed in the dict.**

| field | before | after |
| :--- | :--- | :--- |
| `data` | (40, 522868) | (40, 254, 314) |
| `timebase` | (522868,) | (254,) |
| `nSamp` | 522868 | 254 |
| `refChanTS` | (522868,) | (1, 254, 314) |
| `nBaselineSamp` | not there | 0 (new field) |

The most important change besides the shape is the **meaning of `timebase`**. Before, it was the time in the whole recording. Now it runs from 0 to 248.71 ms and it is the time **inside a trial**, the same for every trial. In the function this is line 72:

```python
ecog['timebase'] = np.arange(-pre_dur_samp, post_dur_samp) * ecog['sampDur']
```

Because the call used `pre_dur_samp = 0`, the timebase starts at 0 and no samples before the movement onset were taken. If a baseline had been asked for, the timebase would start with negative values and **0 would always be the moment the movement starts**. That is the whole point: after the segmentation the time axis is relative to the gesture and not to the recording, and that is what makes it possible to compare or average the trials with each other.

`refChanTS` was cut in the same way (lines 49 to 58 of the function), so the reference still matches the trials, and `nBaselineSamp` was added so that later steps know how many samples of the trial come before the onset.

One detail: there are **314** trials and not 315. One epoch was lost in Session 2 when `remove_bad_epochs` was applied, so `epoch2.pkl` contains 314 onsets and the segmentation simply follows that number.

<h3 style="color: #FF0000; font-weight: bold;">Finish the following code cell</h3>
Choose a random trial between 1 and length(epoch.label) and a random channel between 1 and 40 - try to avoid previously rejected channels!

**Note:**  [np.random.randint](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randint.html)

In [ ]:
# Randomly select a channel and a trial
np.random.seed(0)                                            # so I always get the same example

# pick only from the good channels, so the rejected ones are avoided automatically
select_channel = int(np.random.choice(ecog['selectedChannels']))
select_trial   = int(np.random.randint(1, len(epoch['label']) + 1))

print(f"channel {select_channel}, trial {select_trial} of {len(epoch['label'])}")

# Extract the data for the selected channel and trial
data = np.squeeze(ecog['data'][select_channel - 1,:, select_trial - 1])
data.shape


In [ ]:
# Randomly select a channel and a trial
np.random.seed(0)                                            # so I always get the same example

# pick only from the good channels, so the rejected ones are avoided automatically
select_channel = int(np.random.choice(ecog['selectedChannels']))
select_trial   = int(np.random.randint(1, len(epoch['label']) + 1))

print(f"channel {select_channel}, trial {select_trial} of {len(epoch['label'])}")

# Extract the data for the selected channel and trial
data = np.squeeze(ecog['data'][select_channel - 1,:, select_trial - 1])
data.shape


# Extract the data for the selected channel and trial
data = np.squeeze(ecog['data'][select_channel - 1,:, select_trial - 1])
data.shape

In [ ]:
# Nyquist frequency
nyquist_freq = 1000 / ecog['sampDur'] / 2

# Design a bandpass filter: Order 3, bandpass (0.3-200 Hz)
b, a = butter(3, [0.3 / nyquist_freq, 200 / nyquist_freq], btype='bandpass')

# Apply the filter
data_filtered = filtfilt(b, a, data)

---
<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (Discussion, 1pt):</h2>

* What is the Nyquist Frequency? 

* Why do we need it here for the butterworth filter design? 

* Use the `plot_spectrum` method in the class `SignalFilterAnalyzer` to visualize the results of the filter.

*(Look at a few different trials and change the filter order to see the influence) Are you satisfied with the results of the filter?*

In [ ]:
# --- TASK 3: Nyquist frequency and how good the filter really is ---
srate = 1000 / ecog['sampDur']
nyquist_freq = srate / 2
print("sampling rate = 1000 / sampDur = %.2f Hz" % srate)
print("Nyquist       = srate / 2      = %.2f Hz" % nyquist_freq)
print("cutoffs 0.3 Hz and 200 Hz become %.6f and %.4f after dividing by Nyquist"
      % (0.3 / nyquist_freq, 200 / nyquist_freq))

trial_ms = ecog['nSamp'] / srate * 1000
print("\none trial is %d samples = %.1f ms" % (ecog['nSamp'], trial_ms))
print("one cycle of 0.3 Hz lasts %.2f s, that is %.0f times longer than a whole trial"
      % (1 / 0.3, (1 / 0.3) / (trial_ms / 1000)))

# Use the class on the single trial that was selected above
analyzer = SignalFilterAnalyzer(data=data, fs=srate)
data_filtered = analyzer.apply_filter(order=3, cutoff=[0.3, 200], btype='bandpass',
                                      filter_method='filtfilt')
analyzer.plot_spectrum(log_scale=True)
analyzer.plot_filter_characteristics()

# Does it matter whether I filter first and cut afterwards, or the other way round?
with open(os.path.join(main_path, 'data/raw/ecogStruct2.pkl'), 'rb') as f:
    ecog_cont = pickle.load(f)
cont = np.array(ecog_cont['data'])[select_channel - 1]
onset = epoch['OnsetIdx'][select_trial - 1]

b, a = butter(3, [0.3 / nyquist_freq, 200 / nyquist_freq], btype='bandpass')
filter_then_cut = filtfilt(b, a, cont)[onset:onset + ecog['nSamp']]
cut_then_filter = data_filtered

print("\nchannel %d, trial %d" % (select_channel, select_trial))
print("raw segment     : mean %8.3f   std %6.3f" % (data.mean(), data.std()))
print("filter, then cut: mean %8.3f   std %6.3f" % (filter_then_cut.mean(), filter_then_cut.std()))
print("cut, then filter: mean %8.3f   std %6.3f" % (cut_then_filter.mean(), cut_then_filter.std()))
diff = cut_then_filter - filter_then_cut
print("difference between the two: rms %.3f, and the signal itself only has std %.3f"
      % (np.sqrt((diff ** 2).mean()), filter_then_cut.std()))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

**What is the Nyquist frequency?**

The Nyquist frequency is **half of the sampling rate**. Our data is sampled with `1000 / sampDur = 1017.25 Hz`, so the Nyquist frequency is **508.63 Hz**. It is the highest frequency that can still be represented with this sampling rate. A signal needs at least two samples per cycle to be recognised, so everything faster than 508.63 Hz cannot be measured any more. Worse, it does not simply disappear: it is folded back into the lower frequencies and shows up as a wrong, lower frequency. That is called aliasing, and it cannot be repaired afterwards.

**Why do we need it for the butterworth filter design?**

`butter` does not take the cutoff in Hz. It takes it as a number between 0 and 1, where **1 means the Nyquist frequency**. So the cutoffs have to be divided by it first:

- 0.3 Hz / 508.63 Hz = 0.00059
- 200 Hz / 508.63 Hz = 0.3932

That is exactly what `[0.3 / nyquist_freq, 200 / nyquist_freq]` does in the cell above. It also shows immediately that our upper cutoff of 200 Hz is allowed, because 0.3932 is smaller than 1. Asking for a cutoff above the Nyquist frequency would be impossible, and scipy raises an error for it.

There is a second way: `SignalFilterAnalyzer` calls `butter(order, cutoff, btype=btype, fs=self.fs)`, and if `fs` is given, the cutoff may be written directly in Hz. Both ways do the same thing, in the second one scipy divides by the Nyquist frequency itself.

**Are you satisfied with the results of the filter?**

Looking at the spectrum plot the filter does what it should: above 200 Hz the amplitude is clearly reduced and the passband is left alone. But I am **not satisfied with filtering the trials**, for one concrete reason.

A trial is only 254 samples, so **249.7 ms**. One cycle of 0.3 Hz takes 3.33 s, which is **13 times longer than a whole trial**. A piece of data that short simply does not contain that frequency, so the lower cutoff cannot do anything sensible. Instead of removing a slow drift, the filter treats the level of the short segment itself as something to be removed.

The numbers in the cell show it. For my example trial:

| | mean | std |
| :--- | ---: | ---: |
| raw segment | +6.36 | 17.01 |
| filter the whole recording, then cut | +4.37 | 15.57 |
| cut first, then filter the trial | **-23.18** | 15.76 |

Filtering the short trial shifted it down by about 28, while filtering the continuous data left the level almost unchanged. The difference between the two versions has an rms of 27.6, which is **larger than the std of the signal itself (15.6)**. So the error is bigger than the data.

The correct order is therefore **filter the whole continuous recording first, and cut it into trials afterwards**, which is also what the notebook does in the next section, where the data is loaded again and filtered as a whole. Filtering single trials is only useful here to look at one example and understand what the filter does.

About the order: with order 3 the edge at 200 Hz is quite soft. Increasing the order makes it sharper, but as I found in Task 1.1 a higher order rings longer in time, and that hurts exactly when the pieces are as short as 250 ms. For this data an order between 3 and 8 is a reasonable compromise, and the filtering should be done on the continuous data where the length is not a problem.

---
## Our ECoG Data
Filtering the whole data

In [ ]:
# Reload the data
ecog_file = os.path.join(main_path, 'data/raw/ecogStruct2.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

print(ecog.keys())
len(ecog['data'])

In [ ]:
# Nyquist frequency
nyquistFreq = 1000/ecog['sampDur']/2

# Compute filter coefficients a & b
b, a = butter(3, [0.3 / nyquist_freq, 200 / nyquist_freq], btype='bandpass')

# filtfilt operates along columns
tmp = np.array(ecog['data']).T

# Filter the data
ecog['data'] = filtfilt(b,a,tmp).T
ecog['data'].shape

---
## Cut single trials from time series 

In [ ]:
# Cut timeseries in single trials defined by onset values in epoch2.pkl
ecog = ecog_segment_ts(ecog, epoch['OnsetIdx'], 0, round(0.25 * epoch['srate']))

In [ ]:
ecog['data'].shape

---
## Frequency spectrum

To transform the data to frequency space the Spectral Analysis function `multitaper_spectrum.py` is used. The second input to this function has to be a `dict` (params) containing six specific fields (tapers, Fs, fpass, pad, err, trialave). The following Code block you have already seen in Session_02

In [ ]:
import importlib
import src.multitaper_spectrum
importlib.reload(src.multitaper_spectrum)
from src.multitaper_spectrum import multitaper_spectrum
print("reloaded")


In [ ]:
# Set sepctrum analysis parameters
params = {
    'tapers': [3, 5],  # TW(time-bandwidth product)=3, K(number of tapers)=5
    'pad': 0,  # no padding
    'Fs': 1000 / ecog['sampDur'],
    'fpass': [0, 200],
    'err': 0,
    'trialavg': False
}

# Multitaper strectral analysis
ecog['data'] = np.array(ecog['data']) # As data originally saved in a list

# Initialize f and S
f, S = [],[]

n_epochs = ecog['data'].shape[2]
for epo_idx in range(n_epochs):
    ecog_oneEpo = {
        'data': ecog['data'][:,:,epo_idx],
        'sampDur': ecog['sampDur']
    }
    f_tem, S_tem = multitaper_spectrum(ecog_oneEpo, params)
    f.append(f_tem)
    S.append(S_tem)
f = np.array(f)
S = np.array(S)

# Create a new Dict to store multitaper spectral analysis results
periodogram = {
    'trailList': 1,
    'params': params,
    'periodogram':S,
    'centerFrequency':f
}

# update ecog dict
ecog['periodogram'] = periodogram

# Save the spectrual analysised data 
with open("ecogStruct1_processed.pkl", "wb") as file:
    pickle.dump(ecog, file)

# Check again your data
print(ecog.keys())
print(ecog['periodogram'].keys())
for key, value in ecog['periodogram'].items():
    print(f"Key: {key}, Type: {type(value).__name__}")

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 (2 pt):</h2>

Have a look into the function `multitaper_spectrum` function.

- What does the `structure` `params` contain? 

- What is the multi-taper spectral estimation method and what are tapers? 

- What does the definition ecog.periodogram.params.pad = 1 lead to? What happens if pad is -1 or 0?

- What is padding? Why it is necessary? 

- What are the outputs `[s,f]` going to contain?

- And where will this information be stored in the `ecog` `dictionary`?


In [ ]:
# --- TASK 4: what is really in params and in the output? ---
import math

N  = ecog['nSamp']
Fs = params['Fs']
TW, K = params['tapers']

print("params:")
for key, value in params.items():
    print(f"   {key:9} = {value}")

print("\none trial has %d samples = %.1f ms at %.2f Hz" % (N, N / Fs * 1000, Fs))
print("frequency resolution df = Fs / nfft = %.3f Hz" % (Fs / N))
print("tapers: TW = %d, K = %d, and the largest sensible K is 2*TW-1 = %d" % (TW, K, 2 * TW - 1))
print("the tapers smooth the spectrum over +/- TW*Fs/N = %.2f Hz" % (TW * Fs / N))

print("\noutputs of multitaper_spectrum for ONE trial:")
print("   f (centerFrequency) :", np.array(f).shape if np.ndim(f) == 1 else np.array(f)[0].shape,
      "-> the frequencies, %.2f to %.2f Hz" % (np.array(f).min(), np.array(f).max()))
print("   S (periodogram)     : (n_frequencies, n_channels) = (%d, %d)" % (len(np.array(f).ravel()[:50]), ecog['data'].shape[0]))
print("\nstacked over all trials and stored in the dict:")
print("   ecog['periodogram']['periodogram']    :", np.array(ecog['periodogram']['periodogram']).shape)
print("   ecog['periodogram']['centerFrequency']:", np.array(ecog['periodogram']['centerFrequency']).shape)
print("   ecog['periodogram']['params']         :", type(ecog['periodogram']['params']).__name__)

# what the pad values would mean, and what this implementation actually does
p2 = 2 ** math.ceil(math.log2(N))
print("\npadding, if it were implemented the way Chronux does it:")
print("   pad = -1 -> nfft = %3d  ->  df = %.3f Hz" % (N, Fs / N))
print("   pad =  0 -> nfft = %3d  ->  df = %.3f Hz" % (p2, Fs / p2))
print("   pad =  1 -> nfft = %3d  ->  df = %.3f Hz" % (2 * p2, Fs / (2 * p2)))
print("but line 100 of multitaper_spectrum.py is  nfft = len(ecog['data'][0]),")
print("so nfft is always %d here and params['pad'] is never used." % N)

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

**What does `params` contain?**

| key | value here | meaning |
| :--- | :--- | :--- |
| `tapers` | `[3, 5]` | `[TW, K]`: the time-bandwidth product and the number of tapers |
| `pad` | `0` | how much the data should be padded before the FFT |
| `Fs` | 1017.25 | the sampling rate in Hz, computed as `1000 / sampDur` |
| `fpass` | `[0, 200]` | the frequency range we want to keep |
| `err` | `0` | whether error bars should be computed, this is not implemented in the python version |
| `trialavg` | `False` | whether the result should be averaged over the channels/trials |

**What is the multitaper method and what are tapers?**

If I just take the FFT of a trial, I cut a 249.7 ms piece out of a longer signal. The signal does not go to zero at the two ends, and this sharp cut spreads energy from one frequency into all the neighbouring ones. That is called spectral leakage. A **taper** is a window that is multiplied with the data before the FFT so that it goes smoothly to zero at both ends, which reduces the leakage.

The problem with using one taper is that it throws away the data at the beginning and the end of the trial, so the estimate becomes noisy. The **multitaper method** solves this by using **several different tapers** which are orthogonal to each other, the so called DPSS or Slepian sequences. Each taper gives its own spectrum, and these are averaged. Because the tapers are orthogonal, the single estimates are almost independent, so averaging them reduces the noise instead of just repeating the same error.

The price is that the spectrum gets smoothed. With `TW = 3` and `N = 254` samples, the smoothing is over `TW * Fs / N = 12.01 Hz` to each side. `K` cannot be chosen freely either: the useful maximum is `2*TW - 1 = 5`, and the seminar uses exactly `K = 5`. A larger `K` would add tapers that no longer fit into that bandwidth and would only make the estimate worse.

**What is padding and why is it necessary?**

Padding means adding zeros at the end of the signal before the FFT. It does **not** add information, and it does not improve the real resolution, which is fixed by the length of the recording. What it does is compute the spectrum on a finer frequency grid, so the curve looks smoother and a peak between two bins can be seen better. The other reason is speed: the FFT is fastest when the length is a power of 2.

Our trial has 254 samples and `df = Fs/nfft = 4.005 Hz`. In Chronux the `pad` value works like this:

| pad | nfft | resolution |
| :--- | ---: | ---: |
| -1 | 254 (no padding) | 4.005 Hz |
| 0 | 256 (next power of 2) | 3.974 Hz |
| 1 | 512 (that times 2) | 1.987 Hz |

So `pad = 1` would double the number of frequency points, `pad = 0` only rounds up to the next power of two, and `pad = -1` means no padding at all.

**However, in this python version padding is not implemented.** Line 100 of `multitaper_spectrum.py` is

```python
nfft = len(ecog['data'][0])
```

so `nfft` is always the trial length, and `params['pad']` is never read anywhere in the function. Changing it to 1, 0 or -1 gives exactly the same result here. The key is kept only because the parameter structure follows the original Chronux code.

**What do the outputs contain and where are they stored?**

The function returns `f, S` (the notebook writes `[s, f]`, but the order in the code is frequency first).

- **`f`** is the frequency vector. With `fpass = [0, 200]` it has **50 entries**, from 0 to 196.24 Hz in steps of 4.005 Hz.
- **`S`** is the power spectrum, with the shape **(50 frequencies, 40 channels)** for one trial. Because `trialavg` is `False`, nothing is averaged away.

The cell loops over all 314 trials and stacks the results, so the stored spectrum has the shape **(314, 50, 40)**, meaning trials by frequencies by channels. Everything is then put into a new sub-dictionary inside `ecog`:

```
ecog['periodogram']['periodogram']     -> S, the spectra
ecog['periodogram']['centerFrequency'] -> f, the frequencies
ecog['periodogram']['params']          -> the params dict, so the settings are stored with the result
ecog['periodogram']['trailList']       -> 1
```

Keeping `params` next to the result is useful, because a spectrum on its own cannot be interpreted without knowing which tapers and which sampling rate produced it.